In [0]:
import pandas as pd

In [0]:
sales_df = spark.read.table("live_nation_prod.bronze_prod.ticket_sales")
events_df = spark.read.table("live_nation_prod.silver_prod.events_silver")

In [0]:
sales_df = sales_df.toPandas()
events_df = events_df.toPandas()

In [0]:
sales_df.info()

In [0]:
sales_df.head()

In [0]:
events_df.info()

In [0]:
events_df.nunique() #Even though the venues and dates are the name number of unique, I am not going to assume that there can never be multiple events at the same venue on different days.  

In [0]:
sales_df = sales_df.groupby('event_id').agg(
    {'tickets_sold_in_minute': 'count'}
)

In [0]:
sales_df

In [0]:
sales_df = sales_df.rename(columns={'tickets_sold_in_minute': 'tickets_sold'}).reset_index()

In [0]:
sales_df

In [0]:
merged_df = events_df.merge(sales_df, on='event_id', how='left')

In [0]:
merged_df.head()

In [0]:
merged_df.info()

In [0]:
merged_df.describe()

In [0]:
merged_df['tickets_sold'].plot(kind='hist')

In [0]:
# I used the median over the mean because the data has a large spread
merged_df['tickets_sold'] = merged_df['tickets_sold'].fillna(merged_df['tickets_sold'].median())

In [0]:
merged_df.info()

In [0]:
merged_df = merged_df.sort_values('tickets_sold', ascending = False)

In [0]:
event_totals = merged_df.groupby(['venue_id','event_dt']).agg({'tickets_sold': 'sum'}).reset_index().sort_values('tickets_sold', ascending = False)

In [0]:
event_totals.describe()

In [0]:
event_totals.head()

In [0]:
event_totals.tail()

In [0]:
event_totals[event_totals['tickets_sold'] <29]

The worst performing events are those with the lowest ticket sales. In this case, seven events sold 28 or fewer tickets. The event with venue_id 1365 sold the least, with 27 tickets. All other listed events sold 28 tickets each.

The event at venue_id 99308 on 9/25/26 sold 622 tickets, making it the best performer. The event at venue_id 369613 on 11/3/26 sold 530 tickets, ranking second.